# MACARON: MAthematical Crossover And Recombination Outcome Nexus

In [1]:
%display latex

## Things that bother me about our current model

- Noise gets in the way of noticing patterns
    - EXAMPLE: why did it take us so long to notice the edge-effects on local interference? straight line vs not straight line should have been easy to see from the beginning
- CROISSANT (our simulation software) makes a great "oracle" that we can ask questions to, but as a model it doesn't actually suggest anything about the crossover process, or how interference affects it
- Once we get into non-uniform recombination rates, I don't think the model can easily recover the effects of interference
    - Alternatively: finding out what interference pattern exists in experimental data would require sweeping the entire parameter space via simulation, which could be almost impossible to do comprehensively (need to check different interference schemes, strengths, base recombination rate across chromosome, base average CO rate per chromosome, etc)
- We don't know how to implement assurance

## An analytical probabilistic model of crossovers

What if we modelled every locus as a random variable?

For a chromosome with $n$ loci, define $C = [L_1, ..., L_n], L_i \sim \operatorname{Bern}(p)$ where a success indicates a crossover at locus $i$, and a failure indicates no crossover. Then by definition $P(L_i = 1) = p$ is the probability that there is a crossover at locus $i$ in a given meiosis.

We then need to determine $p$ for each locus. We know that this "total" probability of a CO at a locus is made up of P(CO at $i$) when it is the first crossover (under no interference), plus P(CO at $i$) under various conditions where there is interference.

Starting with the first situation, where there is no interference: Define a function $B: [ 1..n ] \to [0, 1]$ that determines the "base" recombination rate at that locus. That is, under no interference, $P(L_i = 1) = B(i)$.

For example, a uniform recombination curve across the chromosome with an average of 2 COs per chromosome would result in
$$
B(i) = \frac{2}{n}.
$$
This is essentially the base recombination rate we set up right now in CROISSANT.

In [2]:
B_unif(i) = 2 / 10
B_unif

i |--> 1/5

## Representing interference

We can think of interference as a dependence between loci: given a crossover at locus $j$, we have additional information about whether there is a crossover at locus $k$ (a greater or lesser probability, depending on the type of interference). Then we can represent interference via *conditional probabilities*. The probability of a CO at locus $i$ given that there is a crossover at locus $j$ is $P(L_i = 1 | L_j = 1)$.


Then an interference scheme can be represented as $M: \{ (i, j) \mid i, j \in [ 1 .. n ] \subset \mathbb{Z}^+ \} \to \mathbb{R}$, for example local interference:
$$
M(i, j) = 1 - e^{-|\frac{j}{n} - \frac{i}{n}|}
$$

## Analysing recombination using this model: toy example

My litmus test for whether this model is actually useful is to check that we can recapitulate all of our current results - mainly the 4 plots we usually look at: recombination rate, CO-per-chromosome distribution, interevent distance distribution, and cM-RF curves.

We start by attempting to recreate these, using a 10-locus long chromosome. Setting up our random variables:

In [3]:
p = var('p_1 p_2 p_3 p_4 p_5 p_6 p_7 p_8 p_9 p_10')
omega = list(range(1, 11))
pmf = { k: p[k - 1] for k in omega }
pspace = DiscreteProbabilitySpace(omega, pmf)

In [4]:
p_1

p_1

In [5]:
p_L1 = function('p_{L1}')(x)
p_L1
L_1 = DiscreteRandomVariable(pspace, p_L1)
L_1

Discrete random variable defined by p_{L1}(x)

We set up a base recombination rate $B_{\text{unif}, 1}(i)$ representing a uniform recombination rate with an average of 1 CO per chromosome:

In [6]:
B_unif(i) == 1 / 10
B_unif

i |--> 1/5

And a modifier function for local interference $M_\text{local}(i, j)$:

In [7]:
M_local(i, j) = 1 - e^(-abs(j/10 - i/10))
M_local

(i, j) |--> -e^(-abs(-1/10*i + 1/10*j)) + 1

We attempt to create a recombination rate curve. Then for every locus $L_i \sim \operatorname{Bern}(p_i)$, we need to determine $p_i$. By the law of total probability, this is

$$
\begin{align*}
P(L_i = 1) &= P(L_i = 1|L_j = 0, j \in \{ 1..n \}) \cdot P(L_j = 0, j \in \{ 1..n \}) \\
    &+ P(L_i = 1|L_1 = 1, L_j = 0, j \in \{ 2..n \}) \cdot P(L_j = 0, j \in \{ 1..n \}) \\
    &+ P(...) \\
\end{align*}
$$

or

$$
P(L_i = 1) = \sum_{(k_1 ... k_n), k_i \in \{ 0, 1 \}} [
    P(L_i = 1|L_j = k_j, 1 \leq j \leq n, j \neq i) \cdot P(L_j = k_j, 1 \leq j \leq n, j \neq i)
].
$$

In the context of our given interference scheme, this is

$$
P(L_i = 1) = \sum_{(L_1 = k_1 ... L_n = k_n), k_i \in \{ 0, 1 \}} B(i) \cdot \Pi_{\{ j \mid L_j = 1 \}} M(i, j)
$$

In [8]:
M_local(i, j) = 1 - e^(-abs(j/10 - i/10))
M_local

(i, j) |--> -e^(-abs(-1/10*i + 1/10*j)) + 1

In [9]:
solve(4*x + 3 == 0, x)

[x == (-3/4)]

## Implementing assurance

This model makes it straightforward to implement assurance. Assurance tells us that every chromosome has at least 1 CO. That is, given a locus $i$, if there is no CO at any other locus on that chromosome, then there must be a CO at $i$. Translated into our model:

$$
P(L_i = 1 | L_j = 0, 1 \leq j \leq n) = 1
$$

We just adjust P(CO) at $i$ conditioned on no CO at any other locus to equal 1!

## scratch

solving for marginals in the two-locus toy model:

In [10]:
x1, x2 = var('x1 x2')
x1, x2

(x1, x2)

In [11]:
eq1 = x1 == (1/4)*x2 + (1/2)*(1 - x2)
eq2 = x2 == (1/4)*x1 + (1/2)*(1 - x1)
[eq1, eq2]

[x1 == -1/4*x2 + 1/2, x2 == -1/4*x1 + 1/2]

In [12]:
solve([eq1, eq2], [x1, x2])

[[x1 == (2/5), x2 == (2/5)]]

trying to generalise:

In [13]:
var('a b k c d m3')
m3_eq = b + d == m3
m3_eq

b + d == m3

In [14]:
add_to_frac_eq = a/b + k == (a + c)/(b + d)
add_to_frac_eq

k + a/b == (a + c)/(b + d)

In [15]:
add_soln = solve(add_to_frac_eq, k)
add_soln

[k == (b*c - a*d)/(b^2 + b*d)]

In [16]:
mult_to_frac_eq = a/b * k == (a + c)/(b + d)
mult_to_frac_eq

a*k/b == (a + c)/(b + d)

In [17]:
mult_soln = solve(mult_to_frac_eq, k)
mult_soln

[k == (a*b + b*c)/(a*b + a*d)]

In [18]:
p_1_0 = a / b
p_1_1 = c / d
p_1_0, p_1_1

(a/b, c/d)

In [19]:
factor(mult_soln[0]).subs({a/b: p_1_0})

k == (a + c)*b/(a*(b + d))

In [20]:
factor(add_soln[0]).subs({b + d: m3})

k == (b*c - a*d)/(b*m3)